In [7]:
# =============================================================================
# USER SETTINGS
# Edit only this cell
# =============================================================================

# -----------------------------------------------------------------------------
# MARKER
# -----------------------------------------------------------------------------
# Options:
#   "ITS"
#   "16S"
#   "CO1"

MODE = "ITS"


# -----------------------------------------------------------------------------
# SUBSET MODE
# -----------------------------------------------------------------------------
# Options:
#   "headers"
#   "sites"
#   "taxonomy"

SUBSET_MODE = "taxonomy"


# -----------------------------------------------------------------------------
# SUBSET NAME
# -----------------------------------------------------------------------------
# Used to name the output folder.

SUBSET_NAME = "Phallales_subset"


# -----------------------------------------------------------------------------
# FINAL DATABASE
# -----------------------------------------------------------------------------
# Enter ONLY the exact Final Database file name.
#
# The program automatically searches in:
#
# MetaDiv/
# └── output/
#     └── <MODE>/
#         └── FINAL_DB/
#             └── <FINAL_DB_NAME>

FINAL_DB_NAME = "Final_Database_species_only_only_fungi_p1_sppn08.csv"


# -----------------------------------------------------------------------------
# TAXONOMY MODE SETTINGS
# -----------------------------------------------------------------------------
# Used only when:
#
# SUBSET_MODE = "taxonomy"

TAXONOMIC_RANK = "order"

TAXONOMIC_FILTER = "Phallales"

# Options:
#   "exact"
#   "loose"
#   "contains"

TAX_FILTER_MODE = "loose"


# -----------------------------------------------------------------------------
# FILTERING BEHAVIOR
# -----------------------------------------------------------------------------

# Remove biodiversity units whose abundance is zero
# across all retained sites.

DROP_ZERO_OTUS = True


# Remove site/sample columns whose total abundance
# becomes zero after subsetting.

DROP_ZERO_SITES = True


# Recalculate Total_Abundance using only retained sites.

RECALCULATE_TOTAL_ABUNDANCE = True


# -----------------------------------------------------------------------------
# OPTIONAL SITE RENAME
# -----------------------------------------------------------------------------
#
# Example:
#
# Sitio_ABC01
#
# becomes:
#
# ABC01
#
# Use None to disable.

REMOVE_SITE_PREFIX = "Sitio_"

In [8]:
# =============================================================================
# MetaDiv Subset Builder
# =============================================================================
#
# Creates analytical subsets from MetaDiv FINAL_DB files.
#
# SUBSET MODES
#
# headers
#     Select biodiversity units using IDs listed in:
#
#         query_lists/headers.txt
#
# sites
#     Select sample/site columns listed in:
#
#         query_lists/sites.txt
#
# taxonomy
#     Select biodiversity units according to a taxonomic rank.
#
#
# OUTPUT STRUCTURE
#
# subset_outputs/
# └── subset_by_<mode>/
#     └── <SUBSET_NAME>/
#         ├── final_db.csv
#         ├── summary.txt
#         │
#         └── For_R/
#             ├── FOR_R_INFO.txt
#             ├── abundance.csv
#             ├── sample_metadata_to_fill.csv
#             ├── taxonomy.csv
#             └── sequences.fasta
#
# =============================================================================


from pathlib import Path
import pandas as pd
import re
import shutil


# =============================================================================
# 1. DETECT METADIV_SUBSET DIRECTORY
# =============================================================================


def detect_project_dir():

    current_dir = Path.cwd().resolve()

    candidate_dirs = [
        current_dir,
        *current_dir.parents
    ]

    for candidate in candidate_dirs:

        if (
            candidate
            / "query_lists"
        ).is_dir():

            return candidate


    raise FileNotFoundError(
        "\nMetaDiv_subset directory could not be detected.\n\n"
        "The program expects a directory containing:\n\n"
        "MetaDiv_subset/\n"
        "├── MetaDiv_Subset.ipynb\n"
        "└── query_lists/\n"
        "    ├── headers.txt\n"
        "    └── sites.txt\n"
    )


PROJECT_DIR = detect_project_dir()


# =============================================================================
# 2. DETECT MAIN METADIV DIRECTORY
# =============================================================================


def detect_metadiv_root():

    candidate_dirs = [
        PROJECT_DIR,
        *PROJECT_DIR.parents
    ]

    for candidate in candidate_dirs:

        if (
            candidate
            / "output"
        ).is_dir():

            return candidate


    raise FileNotFoundError(
        "\nMetaDiv root directory could not be detected.\n\n"
        "Expected a parent directory containing:\n\n"
        "MetaDiv/\n"
        "└── output/\n"
    )


METADIV_ROOT = detect_metadiv_root()


print("=" * 72)
print("MetaDiv Subset Builder")
print("=" * 72)

print("\nMetaDiv_subset directory:")
print(PROJECT_DIR)

print("\nMetaDiv root:")
print(METADIV_ROOT)


# =============================================================================
# 3. VALIDATE USER SETTINGS
# =============================================================================


VALID_MODES = {
    "ITS",
    "16S",
    "CO1"
}


VALID_SUBSET_MODES = {
    "headers",
    "sites",
    "taxonomy"
}


if MODE not in VALID_MODES:

    raise ValueError(
        "\nMODE must be one of:\n"
        "ITS | 16S | CO1"
    )


if SUBSET_MODE not in VALID_SUBSET_MODES:

    raise ValueError(
        "\nSUBSET_MODE must be one of:\n"
        "headers | sites | taxonomy"
    )


if not SUBSET_NAME:

    raise ValueError(
        "\nSUBSET_NAME cannot be empty."
    )


if not FINAL_DB_NAME:

    raise ValueError(
        "\nFINAL_DB_NAME cannot be empty."
    )


# =============================================================================
# 4. INPUT FILES
# =============================================================================


QUERY_LISTS_DIR = (
    PROJECT_DIR
    / "query_lists"
)


HEADERS_FILE = (
    QUERY_LISTS_DIR
    / "headers.txt"
)


SITES_FILE = (
    QUERY_LISTS_DIR
    / "sites.txt"
)


if not QUERY_LISTS_DIR.is_dir():

    raise FileNotFoundError(
        "\nquery_lists directory was not found:\n"
        f"{QUERY_LISTS_DIR}"
    )


# =============================================================================
# 5. FINAL DATABASE
# =============================================================================


INPUT_FILE = (
    METADIV_ROOT
    / "output"
    / MODE
    / "FINAL_DB"
    / FINAL_DB_NAME
).resolve()


if not INPUT_FILE.exists():

    raise FileNotFoundError(
        "\nFinal Database was not found:\n\n"
        f"{INPUT_FILE}\n\n"
        "Check MODE and FINAL_DB_NAME."
    )


if INPUT_FILE.suffix.lower() != ".csv":

    raise ValueError(
        "\nFINAL_DB_NAME must be a CSV file."
    )


print("\nInput Final Database:")
print(INPUT_FILE)


# =============================================================================
# 6. OUTPUT DIRECTORIES
# =============================================================================


SUBSET_OUTPUTS_DIR = (
    PROJECT_DIR
    / "subset_outputs"
)


MODE_OUTPUT_DIR = (
    SUBSET_OUTPUTS_DIR
    / f"subset_by_{SUBSET_MODE}"
)


OUTPUT_DIR = (
    MODE_OUTPUT_DIR
    / SUBSET_NAME
)


FOR_R_OUTPUT_DIR = (
    OUTPUT_DIR
    / "For_R"
)


# Remove only the current subset folder.
# Other subsets and modes remain untouched.

if OUTPUT_DIR.exists():

    shutil.rmtree(
        OUTPUT_DIR
    )


FOR_R_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 7. CONSTANTS
# =============================================================================


TAX_RANKS = [
    "domain",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species"
]


PVALUE_COLS = [
    f"{rank}_pvalue"
    for rank in TAX_RANKS
]


CORE_ID_COLS = [
    "SPPN",
    "OTU_XX",
    "Original_OTUID",
    "Original_ID",
    "OTU_Number"
]


KNOWN_NON_SITE_COLS = set(
    CORE_ID_COLS
    +
    TAX_RANKS
    +
    PVALUE_COLS
    +
    [
        "sequence",
        "sequence_lenght",
        "sequence_length",
        "Total_Abundance",
        "sintax_taxonomy",
        "primary_lifestyle",
        "Secondary_lifestyle",
        "secondary_lifestyle",
        "functional_group",
        "FAPROTAX_functions",
        "FAPROTAX_groups"
    ]
)


# =============================================================================
# 8. HELPER FUNCTIONS
# =============================================================================


def normalize_text(x):

    return str(
        x
    ).strip().lower()


# -----------------------------------------------------------------------------


def clean_header_id(x):

    x = str(
        x
    ).strip()


    if x.startswith(">"):

        x = x[
            1:
        ].strip()


    return x.split()[0]


# -----------------------------------------------------------------------------


def load_txt_list(file_path):

    file_path = Path(
        file_path
    )


    if not file_path.exists():

        raise FileNotFoundError(
            "\nRequired query-list file was not found:\n"
            f"{file_path}"
        )


    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        values = [
            line.strip()

            for line in f

            if line.strip()
        ]


    if not values:

        raise ValueError(
            "\nQuery-list file is empty:\n"
            f"{file_path}"
        )


    return values


# -----------------------------------------------------------------------------


def detect_id_column(df):

    for col in CORE_ID_COLS:

        if col in df.columns:

            return col


    raise ValueError(
        "\nNo valid biodiversity-unit ID column was found.\n\n"
        "Expected one of:\n"
        +
        ", ".join(
            CORE_ID_COLS
        )
    )


# -----------------------------------------------------------------------------


def detect_site_columns(df):

    site_cols = []


    for col in df.columns:

        col_str = str(
            col
        )


        col_lower = (
            col_str.lower()
        )


        if col in KNOWN_NON_SITE_COLS:

            continue


        if col_lower.startswith(
            "sequence"
        ):

            continue


        if col_lower.endswith(
            "_pvalue"
        ):

            continue


        numeric_values = pd.to_numeric(
            df[
                col
            ],
            errors="coerce"
        )


        non_na_fraction = (
            numeric_values
            .notna()
            .mean()
        )


        if non_na_fraction > 0.80:

            site_cols.append(
                col
            )


    return site_cols


# -----------------------------------------------------------------------------


def rename_sites(
    cols,
    prefix
):

    if not prefix:

        return cols


    renamed = []


    for col in cols:

        if (
            isinstance(
                col,
                str
            )
            and
            col.startswith(
                prefix
            )
        ):

            renamed.append(
                col[
                    len(prefix):
                ]
            )


        else:

            renamed.append(
                col
            )


    return renamed


# -----------------------------------------------------------------------------


def make_safe_sample_names(
    site_cols
):

    safe_names = []

    used = {}


    for original in site_cols:

        name = str(
            original
        ).strip()


        safe = re.sub(
            r"[^A-Za-z0-9_.-]+",
            "_",
            name
        )


        safe = re.sub(
            r"_+",
            "_",
            safe
        ).strip("_")


        if not safe:

            safe = "Sample"


        if safe[0].isdigit():

            safe = (
                f"S_{safe}"
            )


        base = safe


        if base in used:

            used[
                base
            ] += 1


            safe = (
                f"{base}_"
                f"{used[base]}"
            )


        else:

            used[
                base
            ] = 1


        safe_names.append(
            safe
        )


    return safe_names


# -----------------------------------------------------------------------------


def apply_taxonomic_filter(
    df,
    rank,
    filt,
    mode
):

    if rank not in df.columns:

        raise ValueError(
            f"\nTaxonomic rank not found: {rank}"
        )


    if isinstance(
        filt,
        (
            str,
            int,
            float
        )
    ):

        filters = [
            str(
                filt
            )
        ]


    else:

        filters = [
            str(x)

            for x in filt
        ]


    col = df[
        rank
    ].astype(
        str
    )


    if mode == "exact":

        mask = col.isin(
            filters
        )


    elif mode == "loose":

        filters_norm = {
            normalize_text(x)

            for x in filters
        }


        mask = (
            col
            .apply(
                normalize_text
            )
            .isin(
                filters_norm
            )
        )


    elif mode == "contains":

        pattern = "|".join(
            re.escape(x)

            for x in filters
        )


        mask = col.str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )


    else:

        raise ValueError(
            "\nTAX_FILTER_MODE must be:\n"
            "exact | loose | contains"
        )


    out = df[
        mask
    ].copy()


    print(
        "\nTaxonomic filter applied:"
    )

    print(
        f"  Rank   : {rank}"
    )

    print(
        f"  Mode   : {mode}"
    )

    print(
        f"  Filter : {filters}"
    )

    print(
        f"  Rows before: {len(df):,}"
    )

    print(
        f"  Rows after : {len(out):,}"
    )


    return out


# -----------------------------------------------------------------------------


def export_fasta(
    df,
    id_col,
    out_fasta
):

    if (
        id_col not in df.columns
        or
        "sequence" not in df.columns
    ):

        print(
            "\nFASTA export skipped because "
            "ID or sequence column is missing."
        )

        return 0


    n = 0


    with open(
        out_fasta,
        "w",
        encoding="utf-8"
    ) as f:

        for _, row in df.iterrows():

            seq_id = str(
                row[
                    id_col
                ]
            ).strip()


            seq = str(
                row[
                    "sequence"
                ]
            ).strip()


            if (
                seq_id
                and
                seq
                and
                seq_id.lower() != "nan"
                and
                seq.lower() != "nan"
            ):

                f.write(
                    f">{seq_id}\n"
                    f"{seq}\n"
                )


                n += 1


    return n


# =============================================================================
# 9. LOAD FINAL DATABASE
# =============================================================================


df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)


id_col = detect_id_column(
    df
)


print(
    f"\nRows: {len(df):,}"
)

print(
    f"Columns: {len(df.columns):,}"
)

print(
    f"ID column: {id_col}"
)


# =============================================================================
# 10. DETECT SITE COLUMNS
# =============================================================================


all_site_cols = detect_site_columns(
    df
)


print(
    f"\nDetected site columns: "
    f"{len(all_site_cols):,}"
)


# =============================================================================
# 11. APPLY SUBSET MODE
# =============================================================================


if SUBSET_MODE == "headers":

    header_ids = [
        clean_header_id(x)

        for x in load_txt_list(
            HEADERS_FILE
        )
    ]


    subset = df[
        df[
            id_col
        ]
        .astype(
            str
        )
        .isin(
            header_ids
        )
    ].copy()


    selected_site_cols = (
        all_site_cols.copy()
    )


    print(
        f"\nHeader IDs requested: "
        f"{len(header_ids):,}"
    )

    print(
        f"Rows retained: "
        f"{len(subset):,}"
    )


# -----------------------------------------------------------------------------


elif SUBSET_MODE == "sites":

    selected_sites = load_txt_list(
        SITES_FILE
    )


    missing_sites = [
        site

        for site in selected_sites

        if site not in df.columns
    ]


    if missing_sites:

        print(
            "\nMissing sites:"
        )


        for site in missing_sites[
            :30
        ]:

            print(
                "-",
                site
            )


        raise ValueError(
            "\nSome requested sites were not found."
        )


    non_site_cols = [
        col

        for col in df.columns

        if col not in all_site_cols
    ]


    selected_site_cols = (
        selected_sites.copy()
    )


    subset = df[
        non_site_cols
        +
        selected_site_cols
    ].copy()


    print(
        f"\nSelected sites: "
        f"{len(selected_site_cols):,}"
    )


# -----------------------------------------------------------------------------


elif SUBSET_MODE == "taxonomy":

    subset = apply_taxonomic_filter(
        df,
        TAXONOMIC_RANK,
        TAXONOMIC_FILTER,
        TAX_FILTER_MODE
    )


    selected_site_cols = [
        col

        for col in all_site_cols

        if col in subset.columns
    ]


# =============================================================================
# 12. CONVERT ABUNDANCE COLUMNS TO NUMERIC
# =============================================================================


if selected_site_cols:

    subset[
        selected_site_cols
    ] = subset[
        selected_site_cols
    ].apply(
        pd.to_numeric,
        errors="coerce"
    ).fillna(
        0
    )


# =============================================================================
# 13. REMOVE ZERO-ABUNDANCE BIODIVERSITY UNITS
# =============================================================================


if (
    selected_site_cols
    and
    DROP_ZERO_OTUS
):

    before = len(
        subset
    )


    subset = subset[
        subset[
            selected_site_cols
        ].sum(
            axis=1
        )
        > 0
    ].copy()


    after = len(
        subset
    )


    print(
        "\nZero-abundance biodiversity-unit pruning:"
    )

    print(
        f"  {before:,} -> {after:,}"
    )


# =============================================================================
# 14. REMOVE ZERO-ABUNDANCE SITES
# =============================================================================


if (
    selected_site_cols
    and
    DROP_ZERO_SITES
):

    site_sums = subset[
        selected_site_cols
    ].sum(
        axis=0
    )


    zero_sites = site_sums[
        site_sums <= 0
    ].index.tolist()


    if zero_sites:

        subset = subset.drop(
            columns=zero_sites,
            errors="ignore"
        )


        selected_site_cols = [
            col

            for col in selected_site_cols

            if col not in zero_sites
        ]


        print(
            f"\nZero-abundance site columns removed: "
            f"{len(zero_sites):,}"
        )


    else:

        print(
            "\nNo zero-abundance site columns removed."
        )


# =============================================================================
# 15. RECALCULATE TOTAL ABUNDANCE
# =============================================================================


if RECALCULATE_TOTAL_ABUNDANCE:

    if selected_site_cols:

        subset[
            "Total_Abundance"
        ] = subset[
            selected_site_cols
        ].sum(
            axis=1
        )


        print(
            "Total_Abundance recalculated "
            "using retained site columns."
        )


    else:

        subset[
            "Total_Abundance"
        ] = 0


# =============================================================================
# 16. OPTIONAL SITE RENAME
# =============================================================================


site_cols_final = (
    selected_site_cols.copy()
)


if (
    REMOVE_SITE_PREFIX
    and
    site_cols_final
):

    renamed_sites = rename_sites(
        site_cols_final,
        REMOVE_SITE_PREFIX
    )


    rename_map = {
        old: new

        for old, new in zip(
            site_cols_final,
            renamed_sites
        )

        if old != new
    }


    if rename_map:

        subset = subset.rename(
            columns=rename_map
        )


        print(
            f"\nRemoved site prefix: "
            f"{REMOVE_SITE_PREFIX}"
        )


    site_cols_final = (
        renamed_sites
    )


# =============================================================================
# 17. EXPORT MAIN FINAL DATABASE
# =============================================================================


out_final_db = (
    OUTPUT_DIR
    / "final_db.csv"
)


subset.to_csv(
    out_final_db,
    index=False,
    encoding="utf-8"
)


# =============================================================================
# 18. EXPORT FOR_R FILES
# =============================================================================


safe_site_names = make_safe_sample_names(
    site_cols_final
)


safe_rename_map = dict(
    zip(
        site_cols_final,
        safe_site_names
    )
)


# -----------------------------------------------------------------------------
# abundance.csv
# -----------------------------------------------------------------------------


if site_cols_final:

    abundance = subset[
        [id_col]
        +
        site_cols_final
    ].copy()


    for col in site_cols_final:

        abundance[
            col
        ] = pd.to_numeric(
            abundance[
                col
            ],
            errors="coerce"
        ).fillna(
            0
        ).astype(
            "int64"
        )


    abundance = abundance.rename(
        columns=safe_rename_map
    )


else:

    abundance = subset[
        [
            id_col
        ]
    ].copy()


abundance = abundance.drop_duplicates(
    subset=[
        id_col
    ],
    keep="first"
)


abundance = abundance.set_index(
    id_col
)


out_abundance = (
    FOR_R_OUTPUT_DIR
    / "abundance.csv"
)


abundance.to_csv(
    out_abundance,
    encoding="utf-8",
    na_rep="0"
)


# -----------------------------------------------------------------------------
# taxonomy.csv
# -----------------------------------------------------------------------------


tax_cols = [
    col

    for col in TAX_RANKS

    if col in subset.columns
]


pvalue_cols = [
    col

    for col in PVALUE_COLS

    if col in subset.columns
]


ecological_cols = [
    col

    for col in [
        "primary_lifestyle",
        "Secondary_lifestyle",
        "secondary_lifestyle",
        "functional_group",
        "FAPROTAX_functions",
        "FAPROTAX_groups"
    ]

    if col in subset.columns
]


taxonomy_cols = (
    [id_col]
    +
    tax_cols
    +
    pvalue_cols
    +
    ecological_cols
)


taxonomy_cols = list(
    dict.fromkeys(
        taxonomy_cols
    )
)


taxonomy = subset[
    taxonomy_cols
].drop_duplicates(
    subset=[
        id_col
    ],
    keep="first"
)


out_taxonomy = (
    FOR_R_OUTPUT_DIR
    / "taxonomy.csv"
)


taxonomy.to_csv(
    out_taxonomy,
    index=False,
    encoding="utf-8"
)


# -----------------------------------------------------------------------------
# sample_metadata_to_fill.csv
# -----------------------------------------------------------------------------


metadata = pd.DataFrame(
    {
        "SampleID":
            safe_site_names,

        "Group":
            "All_samples",

        "Site":
            safe_site_names
    }
)


out_metadata = (
    FOR_R_OUTPUT_DIR
    / "sample_metadata_to_fill.csv"
)


metadata.to_csv(
    out_metadata,
    index=False,
    encoding="utf-8"
)


# -----------------------------------------------------------------------------
# sequences.fasta
# -----------------------------------------------------------------------------


out_fasta = (
    FOR_R_OUTPUT_DIR
    / "sequences.fasta"
)


fasta_count = export_fasta(
    subset,
    id_col,
    out_fasta
)


# -----------------------------------------------------------------------------
# FOR_R_INFO.txt
# -----------------------------------------------------------------------------


out_for_r_info = (
    FOR_R_OUTPUT_DIR
    / "FOR_R_INFO.txt"
)


with open(
    out_for_r_info,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "MetaDiv Subset Builder - For_R export\n"
    )

    f.write(
        "=====================================\n\n"
    )

    f.write(
        f"Subset name: {SUBSET_NAME}\n"
    )

    f.write(
        f"Subset mode: {SUBSET_MODE}\n"
    )

    f.write(
        f"Marker: {MODE}\n"
    )

    f.write(
        f"Taxa/rows: {len(subset)}\n"
    )

    f.write(
        f"Sites/samples: {len(site_cols_final)}\n"
    )

    f.write(
        f"FASTA sequences: {fasta_count}\n"
    )


# =============================================================================
# 19. SUMMARY.TXT
# =============================================================================


out_summary = (
    OUTPUT_DIR
    / "summary.txt"
)


summary_lines = [
    "MetaDiv Subset Builder summary",
    "==============================",
    "",
    f"MetaDiv root: {METADIV_ROOT}",
    f"MetaDiv_subset directory: {PROJECT_DIR}",
    f"Input FINAL_DB: {INPUT_FILE}",
    f"Marker: {MODE}",
    f"Subset mode: {SUBSET_MODE}",
    f"Subset name: {SUBSET_NAME}",
    "",
    f"Original rows: {len(df):,}",
    f"Subset rows: {len(subset):,}",
    f"Detected original site columns: {len(all_site_cols):,}",
    f"Retained site columns: {len(site_cols_final):,}",
    f"FASTA sequences exported: {fasta_count:,}",
    "",
    "Main outputs:",
    f"- {out_final_db}",
    f"- {out_summary}",
    "",
    "For_R outputs:",
    f"- {out_for_r_info}",
    f"- {out_abundance}",
    f"- {out_metadata}",
    f"- {out_taxonomy}",
    f"- {out_fasta}",
]


if SUBSET_MODE == "headers":

    summary_lines += [
        "",
        f"Header list: {HEADERS_FILE}"
    ]


elif SUBSET_MODE == "sites":

    summary_lines += [
        "",
        f"Site list: {SITES_FILE}"
    ]


elif SUBSET_MODE == "taxonomy":

    summary_lines += [
        "",
        "Taxonomy filter:",
        f"- Rank: {TAXONOMIC_RANK}",
        f"- Filter: {TAXONOMIC_FILTER}",
        f"- Mode: {TAX_FILTER_MODE}"
    ]


with open(
    out_summary,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# =============================================================================
# 20. FINAL SUMMARY
# =============================================================================


print(
    "\n"
    +
    "=" * 72
)


print(
    "DONE"
)


print(
    "=" * 72
)


print(
    "\nSubset mode:"
)

print(
    SUBSET_MODE
)


print(
    "\nCurrent subset folder:"
)

print(
    OUTPUT_DIR
)


print(
    "\nMain files:"
)

print(
    " - final_db.csv"
)

print(
    " - summary.txt"
)


print(
    "\nFor_R files:"
)

print(
    " - FOR_R_INFO.txt"
)

print(
    " - abundance.csv"
)

print(
    " - sample_metadata_to_fill.csv"
)

print(
    " - taxonomy.csv"
)

print(
    " - sequences.fasta"
)


print(
    f"\nFinal biodiversity units: "
    f"{len(subset):,}"
)


print(
    f"Final sites: "
    f"{len(site_cols_final):,}"
)


print(
    f"FASTA sequences exported: "
    f"{fasta_count:,}"
)


print(
    "\n✅ MetaDiv Subset Builder completed successfully."
)

MetaDiv Subset Builder

MetaDiv_subset directory:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\MetaDiv_Utilities\MetaDiv_subsets

MetaDiv root:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN

Input Final Database:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\output\ITS\FINAL_DB\Final_Database_species_only_only_fungi_p1_sppn08.csv

Rows: 41,929
Columns: 348
ID column: SPPN

Detected site columns: 326

Taxonomic filter applied:
  Rank   : order
  Mode   : loose
  Filter : ['Phallales']
  Rows before: 41,929
  Rows after : 32

Zero-abundance biodiversity-unit pruning:
  32 -> 32

Zero-abundance site columns removed: 210
Total_Abundance recalculated using retained site columns.

DONE

Subset mode:
taxonomy

Current subset folder:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\MetaDiv_Utilities\MetaDiv_subsets\subset_outputs\subset_by_taxonomy\Phallales_subset

Main files:
 - final_db.csv
 - summary.txt

For